# Kernel cuantico ZZ para QSVM

Construye el kernel de fidelidad ``K(x_i, x_j) = P(00...0)`` del circuito
``U(x_j)^dagger U(x_i)``, con el feature map ZZ definido en pytket y ejecutado via guppy.

Flujo: cargar datos -> inspeccionar circuitos (sin shots) -> construir la matriz
kernel. Toda la logica vive en `funciones_nexus.py`; aqui solo quedan los
parametros y las llamadas.

La construccion de la matriz esta apagada por defecto (`RUN_MATRIX = False`):
revisa circuitos y costo antes de encenderla.

## 1. Configuracion y datos

Carga el dataset escalado y lo separa en train/test segun `_PartInd_`.

In [ ]:
import pandas as pd
from IPython.display import display
from pytket.circuit.display import render_circuit_jupyter

from funciones_nexus import (
    cargar_datos_kernel,
    zz_feature_map,
    seleccionar_par_kernel,
    iniciar_matriz_kernel,
    consultar_matriz_nexus,
    guardar_kernel_qsvm,
    MATRIX_BACKEND_OPTIONS,
)

PROJECT_NAME = "prueba_migracion"                       # Proyecto de Nexus (se crea si no existe)
KERNEL_DATA_PATH = "data/processed/df_escalado.csv"     # Dataset escalado con _PartInd_

kernel_df, kernel_feature_columns, kernel_train_df, kernel_test_df = cargar_datos_kernel(KERNEL_DATA_PATH)
print("Features:", kernel_feature_columns)
print(f"Train: {kernel_train_df.shape} | Test: {kernel_test_df.shape}")
display(kernel_train_df.head())

## 2. Inspeccion del feature map U(x)

No consume shots.

In [ ]:
PREVIEW_ROW = 0     # Cambia esta fila para inspeccionar otro U(x), sin ejecutar shots

preview_x = kernel_train_df.iloc[PREVIEW_ROW].to_numpy(dtype=float)
feature_map_preview = zz_feature_map(preview_x)
print(f"Feature map de train[{PREVIEW_ROW}] | qubits: {feature_map_preview.n_qubits} | puertas: {feature_map_preview.n_gates}")
render_circuit_jupyter(feature_map_preview)

## 3. Seleccion e inspeccion del par

Construye ``U(x_j)^dagger U(x_i)`` con barreras para revisarlo antes de ejecutar.

In [ ]:
KERNEL_ROW_I = 0    # Filas de train que forman el par
KERNEL_ROW_J = 1

kernel_x_i, kernel_x_j, kernel_preview_circuit = seleccionar_par_kernel(
    kernel_train_df, KERNEL_ROW_I, KERNEL_ROW_J
)
render_circuit_jupyter(kernel_preview_circuit)

## 4. Matriz kernel

Ejecuta el triangulo superior y refleja por simetria (`K(i,j) = K(j,i)`).
Con la diagonal desactivada se fija `K(i,i) = 1` sin ejecutar circuitos:
para `m` filas se requieren `m(m-1)/2` circuitos.

Backends: Selene local o Nexus (Selene, H1/H2 via compile job de pytket, Helios).
Tras enviar a Nexus, **no reejecutes esta celda**: usa la celda de consulta.

In [ ]:
MATRIX_ROWS = [0, 1, 2, 3]                    # Filas de train que forman la matriz
MATRIX_BACKEND = "H1-1LE"   # Ver MATRIX_BACKEND_OPTIONS
RUN_MATRIX = False                          # Interruptor de seguridad
MATRIX_SHOTS = 1000
MATRIX_SEED = 42
MATRIX_EXECUTE_DIAGONAL = True               # False: fija K(i,i)=1 sin ejecutar
SAVE_MATRIX_RUN = True

matrix_state, matrix_result = iniciar_matriz_kernel(
    kernel_train_df, MATRIX_ROWS, MATRIX_BACKEND, RUN_MATRIX,
    n_shots=MATRIX_SHOTS, seed=MATRIX_SEED,
    ejecutar_diagonal=MATRIX_EXECUTE_DIAGONAL,
    guardar=SAVE_MATRIX_RUN, project_name=PROJECT_NAME,
)

if matrix_result is not None:
    display(pd.DataFrame(matrix_result["kernel_matrix"], index=MATRIX_ROWS, columns=MATRIX_ROWS))
    display(matrix_result["run_summary"])

## 5. Consulta de la matriz remota

Reejecutar solo esta celda. Para H1/H2 encadena compile -> execute automaticamente;
al completarse reconstruye la matriz y guarda el CSV.

In [ ]:
matrix_state, matrix_result_remoto = consultar_matriz_nexus(matrix_state, guardar=SAVE_MATRIX_RUN)

if matrix_result_remoto is not None:
    display(pd.DataFrame(matrix_result_remoto["kernel_matrix"], index=MATRIX_ROWS, columns=MATRIX_ROWS))
    display(matrix_result_remoto["run_summary"])

## 6. Guardado del kernel para QSVM

Persiste la matriz de Gram **cuadrada** (lista para `SVC(kernel="precomputed")`) y un CSV de metadatos con su procedencia
(backend, job_id, shots, filas, timestamp). Toma la matriz local o la remota, la que este disponible.

In [ ]:
# Guarda la matriz de Gram cuadrada + metadatos (data/runs/kernel_qsvm_*.csv).
if matrix_result_remoto is not None:
    resultado_final, fuente, id_job = matrix_result_remoto, f"nexus_{MATRIX_BACKEND}", matrix_state["job_ref"].id
elif matrix_result is not None:
    resultado_final, fuente, id_job = matrix_result, "local_statevector", None
else:
    resultado_final = None
    print("Aun no hay una matriz kernel construida que guardar. Corre el paso 4 (o 5).")

if resultado_final is not None:
    ruta_kernel, ruta_meta = guardar_kernel_qsvm(resultado_final, source=fuente, job_id=id_job)
    print("Matriz kernel guardada en:", ruta_kernel)
    print("Metadatos en:", ruta_meta)
    display(pd.read_csv(ruta_kernel, sep=";", index_col=0))